# Task 2 — Electricity Price Prediction

This project focuses on predicting Swiss electricity prices from heterogeneous tabular data containing both numerical and categorical features as well as missing values.

The data is preprocessed using median/mode imputation, feature standardization, and one-hot encoding. A Gaussian Process Regression model with an RBF kernel is then trained to predict the target variable `price_CHF`.

ETH Zürich — Introduction to Machine Learning, Spring 2024

First, we import necessary libraries:

In [149]:
import numpy as np
import pandas as pd
# Add any other imports you need here
    #For preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

    #For kernal and regression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, DotProduct
from sklearn.model_selection import GridSearchCV
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, DotProduct, Matern, RationalQuadratic


# Data Loading
TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
(and potentially change initialization of variables to accomodate how you deal with non-numeric data)

In [150]:
train_df = pd.read_csv("train.csv")
#print(train_df.head(10))
train_df_clean = train_df.dropna(subset=['price_CHF'])
#print(train_df_clean.head(10))
X_train = train_df_clean.drop(['price_CHF'], axis=1)
#print(X_train)
y_train = train_df_clean['price_CHF']
print(y_train)

0      9.644028
1      7.246061
2      7.620085
3      8.411894
4      8.926884
         ...   
889    3.237347
890    2.679221
891    3.633928
894   -0.303802
896   -1.734754
Name: price_CHF, Length: 631, dtype: float64


In [151]:
"""
This loads the training and test data, preprocesses it, removes the NaN
values and interpolates the missing data using imputation

Parameters
----------
Compute
----------
X_train: matrix of floats, training input with features
y_train: array of floats, training output with labels
X_test: matrix of floats: dim = (100, ?), test input with features
"""
# Load training data
train_df = pd.read_csv("train.csv")
    
#print("Training data:")
#print("Shape:", train_df.shape)
#print(train_df.head(2))
#print('\n')
    
# Load test data
test_df = pd.read_csv("test.csv")

#print("Test data:")
#print(test_df.shape)
#print(test_df.head(2))


#Dropping all rows with Na as argument in the price_CHF coloum
train_df_clean = train_df.dropna(subset=['price_CHF'])


# Split into features and target
X_train = train_df_clean.drop(['price_CHF'], axis=1) #Creating the traing set (Attributes)
y_train = train_df_clean['price_CHF'] #Creating the training set (Price for electricity)
X_test = test_df.copy() #Making a copy of the test_set for testing


# Preprocessing for numerical data
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns #Choosing all coloums/titles with numbers in the coloumn
print(numeric_features)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Preprocessing for categorical data
categorical_features = X_train.select_dtypes(include=['object']).columns #Choosing all coloums/titles with strings/objects in the coloumn
print(categorical_features)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

# Apply preprocessing
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

assert (X_train_preprocessed.shape[1] == X_test_preprocessed.shape[1]) and (X_train_preprocessed.shape[0] == y_train.shape[0]) and (X_test_preprocessed.shape[0] == 100), "Invalid data shape"

Index(['price_AUS', 'price_CZE', 'price_GER', 'price_ESP', 'price_FRA',
       'price_UK', 'price_ITA', 'price_POL', 'price_SVK'],
      dtype='object')
Index(['season'], dtype='object')


# Modeling and Prediction
TODO: Define the model and fit it using training data. Then, use test data to make predictions

In [160]:
# Define the mixed kernel
large_kernel = (
    ConstantKernel(constant_value=1.0, constant_value_bounds=(1e-8, 1e2)) *
    RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2)) *
    DotProduct(sigma_0=1.0) *
    Matern(length_scale=1.0, length_scale_bounds=(1e-10, 1e2)) *
    RationalQuadratic(length_scale=1.0, alpha=0.1)
)

# Hyperparameter search space
param_grid = {
    'kernel': [large_kernel],
    'alpha': [0.01, 0.1, 1.0]  # Values to be tuned
}

In [180]:
# Split the data into training and validation sets
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train_preprocessed, y_train, test_size=0.2, random_state=42)

# Train the Gaussian process regressor with Bayesian model selection
gp_model = GaussianProcessRegressor(kernel=mixed_kernel, alpha=0.01, optimizer='fmin_l_bfgs_b', n_restarts_optimizer=10, random_state=42)
gp_model.fit(X_train_split, y_train_split)

# Make predictions
y_pred = gp_model.predict(X_val)

# Calculate R2 score
r2 = r2_score(y_val, y_pred)
print("R2 Score:", r2)

# Save the predictions to a CSV file
results_df = pd.DataFrame({'y_true': y_val.values.flatten(), 'y_pred': y_pred.flatten()})
results_df.to_csv('predictions.csv', index=False)
print("Predictions saved to predictions.csv")

R2 Score: 0.9518821046931282
Predictions saved to predictions.csv


C:\Users\gianl\anaconda3\envs\introtoml\lib\site-packages\sklearn\gaussian_process\kernels.py:429: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\gianl\anaconda3\envs\introtoml\lib\site-packages\sklearn\gaussian_process\kernels.py:419: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [12]:
"""
This defines the model, fits training data and then does the prediction
with the test data 

Parameters
----------
X_train: matrix of floats, training input with 10 features
y_train: array of floats, training output
X_test: matrix of floats: dim = (100, ?), test input with 10 features

Compute
----------
y_test: array of floats: dim = (100,), predictions on test set
"""

y_pred=np.zeros(X_test.shape[0])
#TODO: Define the model and fit it using training data. Then, use test data to make predictions

kernel = RBF(length_scale=0.1)

# Instantiate a Gaussian Process model
gpr = GaussianProcessRegressor(kernel=kernel, alpha= 0.01, random_state=42)

# Fit the model on the training data
gpr.fit(X_train_preprocessed, y_train)

# Make predictions on the test data
y_pred = gpr.predict(X_test_preprocessed)
print(y_pred)

assert y_pred.shape == (100,), "Invalid data shape"

[-2.6883454  -2.0118656  -2.96992602 -1.74122095 -1.78656258 -1.46832042
 -0.62360421 -0.13675879 -0.61814603  0.81615802  1.57964181  2.55576562
  2.28136429  3.80393312  2.43693204  1.67050948  1.01882778  1.58870709
  3.09336197  2.03416933  3.04092698  3.78023735  3.74988103  2.88241166
  2.38364313  3.98690263  7.30476096  8.08298301  7.83636682  7.67891278
  8.05464017  6.20438444  7.99576815  8.07215022  8.49148605  6.84882886
  7.88419844  8.1175535   8.21944829  6.79480288  7.70091217  8.65683489
  7.66391977  7.45800445  7.90106868  7.338372    7.58214431  9.11823343
  8.81183133  6.46498226  7.93096307  7.55203588  8.24044932  7.60288797
  7.06317987  7.62458596  8.8025581   8.25347631  8.24259087  6.35430275
  6.77151614  5.72274925  4.68744644  5.17557736  4.54356011  5.47769522
  4.75624874  5.31901919  4.66204614  4.94886735  4.84330622  5.09346592
  4.49152663  6.05717708  6.39277675  6.17494905  7.55110309  8.93976942
  8.8346989   8.3305484   7.57844212  8.95500435  7

# Saving Results
You don't have to change this

In [13]:
dt = pd.DataFrame(y_pred) 
dt.columns = ['price_CHF']
dt.to_csv('results.csv', index=False)
print("\nResults file successfully generated!")


Results file successfully generated!
